## Setup: Install and Import

In [14]:
!pip install --upgrade sentence-transformers huggingface_hub --break-system-packages

import os
import pandas as pd
import nltk
from scipy import spatial
from sentence_transformers import SentenceTransformer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # Add this line
nltk.download('averaged_perceptron_tagger', quiet=True)

print("✓ Setup complete!")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✓ Setup complete!


## Configuration

In [ ]:

MITRE_CSV = "/home/jovyan/work/MITREMapping/enterprise-techniques-sub.csv"          # MITRE ATT&CK CSV
OUTPUT_FILE = 'mapped_patterns.csv'              # Output file

THRESHOLD = 1        # Distance threshold (paper optimal: 0.6)
WEIGHT_TITLE = 0.5     # Title weight (paper optimal: 0.4)

# =============================

## Load Functions

In [44]:
# Initialize
bert_model = SentenceTransformer('all-mpnet-base-v2')
embedding_memo = {}

# Load MITRE framework
def load_attack_patterns(csv_path):

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    attack_tree = {}

    current_parent_id = None
    current_parent_name = None
    current_parent_desc = None

    for _, row in df.iterrows():

        parent_id = row["ID"]
        sub_id = row["SubTechID"]
        name = row["Name"]
        desc = row["Description"]

        # ---------------- Parent ----------------
        if pd.notnull(parent_id):

            current_parent_id = parent_id
            current_parent_name = name
            current_parent_desc = desc

            attack_tree[parent_id] = {
                "parent_name": name,
                "parent_description": desc,
                "subtechs": {}
            }

        # ---------------- Sub-tech ----------------
        elif pd.notnull(sub_id) and current_parent_id:

            sub_id = str(sub_id)

            if "." in sub_id:
                sub_id = sub_id.split(".")[-1]

            sub_id = sub_id.zfill(3)

            full_id = f"{current_parent_id}.{sub_id}"

            attack_tree[current_parent_id]["subtechs"][full_id] = {
                "tech_name": name,
                "tech_description": desc,
                "full_text": f"{name}. {desc}"
            }

    return attack_tree

# Embedding functions
def get_embedding(txt):
    if txt in embedding_memo:
        return embedding_memo[txt]
    emb = bert_model.encode([txt])[0]
    embedding_memo[txt] = emb
    return emb


def get_embedding_distance(txt1, txt2):
    p1 = get_embedding(txt1)
    p2 = get_embedding(txt2)
    return spatial.distance.cosine(p1, p2)


def hierarchical_weighted_parent_only(
    text,
    parent_title_emb,
    parent_desc_emb,
    sub_embeddings,
    weight_title=0.4,
    top_k_parents=5
):

    query_emb = get_embedding(text)

    # -----------------------------
    # Phase 1 — Weighted Parent
    # -----------------------------
    parent_scores = []

    for parent_id in parent_title_emb:

        d = (
            weight_title * spatial.distance.cosine(query_emb, parent_title_emb[parent_id])
            + (1 - weight_title) * spatial.distance.cosine(query_emb, parent_desc_emb[parent_id])
        )

        parent_scores.append((parent_id, d))

    parent_scores.sort(key=lambda x: x[1])
    top_parents = parent_scores[:top_k_parents]

    # -----------------------------
    # Phase 2 — Sub-tech (flat)
    # -----------------------------
    best_sub = None
    min_dist = float("inf")

    alpha = 0  # tune between 0.2–0.4
    
    for parent_id, parent_score in top_parents:

        for sub_id, sub_emb in sub_embeddings[parent_id].items():

            d_sub = spatial.distance.cosine(query_emb, sub_emb)

            # 🔥 NEW FUSED SCORE
            final_score = alpha * parent_score + (1 - alpha) * d_sub
            
            if final_score < min_dist:
                min_dist = final_score
                best_sub = sub_id

    return best_sub, min_dist, top_parents


# Text processing
def remove_consec_newline(s):
    if not s:
        return s

    ret = s[0]
    for x in s[1:]:
        if not (x == ret[-1] and ret[-1] == '\n'):
            ret += x
    return ret


def extract_sentences(text):
    text = remove_consec_newline(text)
    text = text.replace('\t', ' ').replace("\'", "'")
    sents_nltk = nltk.sent_tokenize(text)

    sents = []
    for x in sents_nltk:
        sents += x.split('\n')

    return [s.strip() for s in sents if s.strip()]


print("✓ Functions loaded")

Loading weights: 100%|██████████████████████████████████████████████| 199/199 [00:00<00:00, 692.17it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Functions loaded


## Run Mapping

In [45]:
def build_embeddings(attack_tree):

    parent_title_emb = {}
    parent_desc_emb  = {}

    sub_embeddings = {}

    for parent_id, data in attack_tree.items():

        # Split only parent
        parent_title_emb[parent_id] = get_embedding(
            data["parent_name"]
        )

        parent_desc_emb[parent_id] = get_embedding(
            data["parent_description"]
        )

        # Keep sub-tech as full text (original approach)
        sub_embeddings[parent_id] = {}

        for sub_id, sub_data in data["subtechs"].items():

            sub_embeddings[parent_id][sub_id] = get_embedding(
                sub_data["full_text"]
            )

    return parent_title_emb, parent_desc_emb, sub_embeddings

In [46]:
print("Loading MITRE ATT&CK framework...")
attack_tree = load_attack_patterns(MITRE_CSV)
print(f"✓ Loaded {len(attack_tree)} techniques\n")

# Read input file
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    text = f.read()

# -----------------------------
# 1️⃣ Load Ground Truth
# -----------------------------
gt = pd.read_csv("/home/jovyan/work/MITREMapping/mapping-gt-250-2.csv")

print("Ground truth samples:", len(gt))
print("GT Columns:", gt.columns.tolist())

TRUE_COL = "true_subtech"
PHRASE_COL = "phrase"

print("THRESHOLD")
print(THRESHOLD)

print("WEIGHT_TITLE - NOT USED")
print(WEIGHT_TITLE)

mapped = {}
correct_matches = 0
total_predictions = 0



# Precompute embeddings
parent_title_emb, parent_desc_emb, sub_embeddings = build_embeddings(attack_tree)

for i, row in gt.iterrows():

    phrase = row[PHRASE_COL]
    true_parent = row[TRUE_COL]

    _id, dist, top_parents = hierarchical_weighted_parent_only(
        phrase,
        parent_title_emb,
        parent_desc_emb,
        sub_embeddings,
        weight_title=WEIGHT_TITLE,
        top_k_parents=3
    )

    # Extract only parent IDs for display
    top_parent_ids = [p[0] for p in top_parents]

    if dist < THRESHOLD:
        total_predictions += 1

        # Store best example per technique
        if _id not in mapped or dist < mapped[_id][0]:
            mapped[_id] = (dist, phrase)

        is_correct = (_id == true_parent)

        if is_correct:
            correct_matches += 1

        status = "✓" if is_correct else "✗"

        print(
            f"[{i:3d}] {status} "
            f"TRUE: {true_parent} | "
            f"TOP_PARENTS: {top_parent_ids} | "
            f"FINAL_SUB: {_id} | "
            f"Dist: {dist:.3f}"
        )

    else:
        print(
            f"[{i:3d}] - BELOW THRESHOLD | "
            f"TRUE: {true_parent} | "
            f"TOP_PARENTS: {top_parent_ids} | "
            f"BestDist: {dist:.3f}"
        )

print("\n==============================")
print(f"✓ Mapped {len(mapped)} unique techniques")
print(f"✓ Total Predictions: {total_predictions}")
print(f"✓ Correct Matches: {correct_matches}")

if total_predictions > 0:
    accuracy = correct_matches / total_predictions
    print(f"✓ Accuracy (on predicted only): {accuracy:.4f}")

Loading MITRE ATT&CK framework...
✓ Loaded 191 techniques

Ground truth samples: 250
GT Columns: ['phrase_id', 'phrase', 'true_parent', 'true_subtech']
THRESHOLD
0.6
WEIGHT_TITLE - NOT USED
0.5
[  0] ✗ TRUE: T1055.001 | TOP_PARENTS: ['T1218', 'T1055', 'T1127'] | FINAL_SUB: T1218.004 | Dist: 0.492
[  1] ✗ TRUE: T1055.001 | TOP_PARENTS: ['T1574', 'T1055', 'T1218'] | FINAL_SUB: T1574.002 | Dist: 0.401
[  2] ✗ TRUE: T1055.001 | TOP_PARENTS: ['T1047', 'T1543', 'T1055'] | FINAL_SUB: T1543.003 | Dist: 0.539
[  3] ✓ TRUE: T1055.001 | TOP_PARENTS: ['T1055', 'T1574', 'T1218'] | FINAL_SUB: T1055.001 | Dist: 0.389
[  4] ✗ TRUE: T1055.001 | TOP_PARENTS: ['T1574', 'T1055', 'T1620'] | FINAL_SUB: T1574.002 | Dist: 0.391
[  5] ✓ TRUE: T1055.001 | TOP_PARENTS: ['T1055', 'T1218', 'T1574'] | FINAL_SUB: T1055.001 | Dist: 0.401
[  6] ✗ TRUE: T1055.001 | TOP_PARENTS: ['T1218', 'T1127', 'T1216'] | FINAL_SUB: T1218.011 | Dist: 0.487
[  7] ✗ TRUE: T1055.001 | TOP_PARENTS: ['T1620', 'T1129', 'T1218'] | FINAL_SUB